# Phase VIII — Crash Regime Typology

**Purpose**: Formal comparison of Type A (Systemic, GFC) vs Type B (Cyclical, 2022) crash dynamics.

**No new models or thresholds are fitted.** All results derive from the established panel pipeline
and prior-phase analyses (phase_oos_crossmarket_regime_only.ipynb, scripts/oos_2022_raterise.py).

Reference: docs/PHASE_VIII_CRASH_TYPOLOGY.md

In [1]:
# =============================================================================
# CELL 1 — Load panel (same pipeline as Phase VII crossmarket notebook)
# =============================================================================
import sys
from pathlib import Path

_nb_root = Path.cwd()
for _candidate in [_nb_root, _nb_root.parent, _nb_root.parent.parent]:
    if (_candidate / 'src').is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import requests
import numpy as np
import pandas as pd
from io import StringIO

from src.loaders.fred import load_fred_hpi_panel
from src.evaluation.regime import assign_fragility_regime
from src.backtests.evaluation import summarize

try:
    display
except NameError:
    def display(df): print(df.to_string(index=False))

# --- Constants (pre-specified, not fitted) -----------------------------------
REAL_RATE_THRESHOLD = 0.0
CITIES = ['austin', 'phoenix', 'las_vegas', 'miami']

# Analysis windows
GFC_START    = '2006-10-01'   # onset of GFC-era adverse period
GFC_END      = '2013-01-01'
ERA22_START  = '2021-10-01'   # last regime transition quarter
ERA22_END    = None           # latest available

# --- Build panel -------------------------------------------------------------
hpi = load_fred_hpi_panel(cities=CITIES, start='1982-01-01')

url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=REAINTRATREARAT10Y'
rate_raw = pd.read_csv(StringIO(requests.get(url, timeout=30).text))
rate_raw.columns = ['date', 'real_rate']
rate_raw['date'] = pd.to_datetime(rate_raw['date'])
rate_raw['real_rate'] = pd.to_numeric(rate_raw['real_rate'], errors='coerce')
rate_q = (
    rate_raw.set_index('date')
    .resample('QS').mean()
    .reset_index()
    .dropna(subset=['real_rate'])
)

panel = pd.merge(hpi, rate_q[['date', 'real_rate']], on='date', how='inner')
panel = assign_fragility_regime(panel, threshold=REAL_RATE_THRESHOLD)
panel['ret_1q'] = panel.groupby('region')['price'].transform(
    lambda s: np.log(s).diff()
)
panel = panel.sort_values(['region', 'date']).reset_index(drop=True)

gfc_panel = panel[panel['date'].between(GFC_START, GFC_END)].copy()
era22_panel = panel[panel['date'] >= ERA22_START].copy()

print(f'Full panel : {panel.shape}')
print(f'GFC window : {gfc_panel["date"].min().date()} – {gfc_panel["date"].max().date()}')
print(f'2022 window: {era22_panel["date"].min().date()} – {era22_panel["date"].max().date()}')
print(f'\nRegime in GFC window (all cities same):')
gfc_regime = gfc_panel[['date','real_rate','regime']].drop_duplicates('date')
print(f'  regime=0 (adverse)     : {(gfc_regime["regime"]==0).sum()} quarters')
print(f'  regime=1 (accommodative): {(gfc_regime["regime"]==1).sum()} quarters')
print(f'  real_rate range: {gfc_regime["real_rate"].min():.2f}% – {gfc_regime["real_rate"].max():.2f}%')
print(f'\nRegime in 2022 window (all cities same):')
era22_regime = era22_panel[['date','real_rate','regime']].drop_duplicates('date')
print(f'  regime=0 (adverse)     : {(era22_regime["regime"]==0).sum()} quarters')
print(f'  regime=1 (accommodative): {(era22_regime["regime"]==1).sum()} quarters')

Full panel : (704, 8)
GFC window : 2006-10-01 – 2013-01-01
2022 window: 2021-10-01 – 2025-10-01

Regime in GFC window (all cities same):
  regime=0 (adverse)     : 24 quarters
  regime=1 (accommodative): 2 quarters
  real_rate range: -0.09% – 2.18%

Regime in 2022 window (all cities same):
  regime=0 (adverse)     : 17 quarters
  regime=1 (accommodative): 0 quarters


In [2]:
# =============================================================================
# CELL 2 — Side-by-side comparison table: GFC vs 2022
# =============================================================================

def crash_metrics(city_data, period_label, regime_trans_date=None):
    """Compute peak-to-trough drawdown metrics for one city in one period."""
    c = city_data.sort_values('date').reset_index(drop=True)
    prices = c['price'].to_numpy()
    dates  = c['date'].to_numpy()
    n = len(prices)
    if n == 0:
        return None
    peak_i  = int(np.argmax(prices))
    post    = prices[peak_i:]
    trough_i = peak_i + int(np.argmin(post))
    dd = (prices[trough_i] / prices[peak_i] - 1) * 100
    n_q_crash = trough_i - peak_i
    q_to_peak = None
    if regime_trans_date is not None:
        td = pd.Timestamp(regime_trans_date)
        peak_date = pd.Timestamp(dates[peak_i])
        q_to_peak = max(0, (peak_date.year - td.year)*4 + (peak_date.month - td.month)//3)
    return {
        'period':        period_label,
        'peak_date':     str(pd.Timestamp(dates[peak_i]).date()),
        'trough_date':   str(pd.Timestamp(dates[trough_i]).date()),
        'drawdown_pct':  round(dd, 1),
        'q_peak_to_trough': n_q_crash,
        'q_trans_to_peak':  q_to_peak,
    }


rows = []
for city in CITIES:
    gfc_city = gfc_panel[gfc_panel['region'] == city]
    era_city = era22_panel[era22_panel['region'] == city]
    m_gfc = crash_metrics(gfc_city, 'GFC (Type A)', regime_trans_date=None)
    m_22  = crash_metrics(era_city, '2022 (Type B)', regime_trans_date='2021-10-01')
    if m_gfc: m_gfc['region'] = city
    if m_22:  m_22['region']  = city
    if m_gfc: rows.append(m_gfc)
    if m_22:  rows.append(m_22)

comp = pd.DataFrame(rows)[[
    'region', 'period', 'peak_date', 'trough_date',
    'drawdown_pct', 'q_peak_to_trough', 'q_trans_to_peak'
]]
comp.columns = ['city', 'type', 'peak', 'trough', 'drawdown_%', 'Q_peak→trough', 'Q_trans→peak']

print('TABLE: GFC vs 2022 Crash Dynamics per City')
print('Note: Q_trans→peak = quarters from last 1→0 regime transition to price peak')
print('      (GFC: no transition — regime was continuously adverse; shown as None)')
print('=' * 90)
display(comp)

# Cross-market return correlation during each period
print()
for lbl, sub in [('GFC (2006Q4–2013Q1)', gfc_panel), ('2022+ (2021Q4–latest)', era22_panel)]:
    wide = sub.pivot(index='date', columns='region', values='ret_1q').dropna()
    corr = wide.corr()
    # Average off-diagonal
    mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
    avg_corr = corr.values[mask].mean()
    print(f'Cross-market return correlation [{lbl}]: avg = {avg_corr:.3f}')
    print(corr.round(3).to_string())
    print()

TABLE: GFC vs 2022 Crash Dynamics per City
Note: Q_trans→peak = quarters from last 1→0 regime transition to price peak
      (GFC: no transition — regime was continuously adverse; shown as None)


,city,type,peak,trough,drawdown_%,Q_peak→trough,Q_trans→peak
0,austin,GFC (Type A),2013-01-01,2013-01-01,0.0,0,NaN
1,austin,2022 (Type B),2022-04-01,2024-10-01,-11.8,10,2.0
2,phoenix,GFC (Type A),2006-10-01,2011-04-01,-51.1,18,NaN
3,phoenix,2022 (Type B),2025-10-01,2025-10-01,0.0,0,16.0
4,las_vegas,GFC (Type A),2006-10-01,2012-01-01,-61.0,21,NaN
5,las_vegas,2022 (Type B),2025-01-01,2025-04-01,-4.3,1,13.0
6,miami,GFC (Type A),2007-04-01,2011-04-01,-47.4,16,NaN
7,miami,2022 (Type B),2025-04-01,2025-07-01,-2.6,1,14.0



Cross-market return correlation [GFC (2006Q4–2013Q1)]: avg = 0.715
region     austin  las_vegas  miami  phoenix
region                                      
austin      1.000      0.467  0.538    0.592
las_vegas   0.467      1.000  0.925    0.906
miami       0.538      0.925  1.000    0.860
phoenix     0.592      0.906  0.860    1.000

Cross-market return correlation [2022+ (2021Q4–latest)]: avg = 0.711
region     austin  las_vegas  miami  phoenix
region                                      
austin      1.000      0.787  0.621    0.911
las_vegas   0.787      1.000  0.473    0.749
miami       0.621      0.473  1.000    0.723
phoenix     0.911      0.749  0.723    1.000



In [3]:
# =============================================================================
# CELL 3 — Two-panel figure: GFC equity curves (left) vs 2022 equity curves (right)
# Both panels: pooled always-in vs regime overlay, with regime shading
# =============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

_here = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd()
_root = _here
for _cand in [_here, _here.parent, _here.parent.parent]:
    if (_cand / 'src').is_dir():
        _root = _cand
        break
OUT_DIR = _root / 'outputs' / 'oos'
OUT_DIR.mkdir(parents=True, exist_ok=True)


def build_pooled_equity(sub, regime_col='regime', ret_col='ret_1q'):
    """Build pooled always-in and overlay equity curves from a panel."""
    # Average quarterly return across cities by date
    agg = (
        sub.groupby('date')
        .agg(ret_mean=(ret_col, 'mean'), regime=('regime', 'first'))
        .reset_index()
        .sort_values('date')
    )
    agg['ret_mean'] = agg['ret_mean'].fillna(0.0)
    agg['overlay_ret'] = agg['ret_mean'].where(agg['regime'] == 1, other=0.0)
    agg['eq_always_in'] = np.exp(agg['ret_mean'].cumsum())
    agg['eq_overlay']   = np.exp(agg['overlay_ret'].cumsum())
    return agg


def add_regime_shading(ax, dates, regimes):
    """Shade accommodative periods (regime=1) on ax."""
    in_accom, start_idx = False, None
    for i, (d, r) in enumerate(zip(dates, regimes)):
        if r == 1 and not in_accom:
            start_idx, in_accom = i, True
        elif r != 1 and in_accom:
            ax.axvspan(dates.iloc[start_idx], dates.iloc[i-1],
                       alpha=0.12, color='green', zorder=0)
            in_accom = False
    if in_accom:
        ax.axvspan(dates.iloc[start_idx], dates.iloc[-1],
                   alpha=0.12, color='green', zorder=0)


gfc_agg  = build_pooled_equity(gfc_panel)
era22_agg = build_pooled_equity(era22_panel)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    'Phase VIII: Crash Regime Typology\n'
    'Type A (Systemic, GFC) vs Type B (Cyclical, 2022) — Pooled Equity Curves',
    fontsize=11, fontweight='bold'
)

# --- Left panel: GFC ---------------------------------------------------------
ax1.plot(gfc_agg['date'], gfc_agg['eq_always_in'],
         color='steelblue', lw=2.0, label='Always-in')
ax1.plot(gfc_agg['date'], gfc_agg['eq_overlay'],
         color='darkorange', lw=2.0, label='Regime overlay')
ax1.axhline(1.0, color='gray', lw=0.8, ls='--', label='Always-out (base=1)')
add_regime_shading(ax1, gfc_agg['date'], gfc_agg['regime'])
ax1.axvline(pd.Timestamp('2008-09-01'), color='red', lw=1.0, ls=':', alpha=0.8)
ax1.text(pd.Timestamp('2008-09-01'), ax1.get_ylim()[0] * 1.02,
         'Lehman', fontsize=7, color='red', ha='left', va='bottom', rotation=90)
ax1.set_title('Type A — Systemic (GFC 2006Q4–2013Q1)', fontsize=9, fontweight='bold')
ax1.set_ylabel('Equity (base=1.0 at start)', fontsize=8)
ax1.set_xlabel('Date', fontsize=8)
ax1.tick_params(labelsize=7)
ax1.grid(True, alpha=0.3)
# Annotation: peak-to-trough
ax1.annotate(
    'Phoenix/LV/Miami\n–47% to –61%\n(16–21 quarters)',
    xy=(pd.Timestamp('2010-01-01'), 0.62), fontsize=7,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8)
)
ax1.annotate(
    'Regime: adverse throughout\n(no 1→0 transition)',
    xy=(pd.Timestamp('2007-07-01'), 1.02), fontsize=7,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8)
)
ax1.legend(fontsize=7, loc='lower left')

# --- Right panel: 2022 -------------------------------------------------------
ax2.plot(era22_agg['date'], era22_agg['eq_always_in'],
         color='steelblue', lw=2.0, label='Always-in')
ax2.plot(era22_agg['date'], era22_agg['eq_overlay'],
         color='darkorange', lw=2.0, label='Regime overlay (exits at 2021Q4)')
ax2.axhline(1.0, color='gray', lw=0.8, ls='--', label='Always-out (base=1)')
add_regime_shading(ax2, era22_agg['date'], era22_agg['regime'])
ax2.axvline(pd.Timestamp('2021-10-01'), color='purple', lw=1.0, ls=':', alpha=0.8)
ax2.text(pd.Timestamp('2021-10-01'), ax2.get_ylim()[0] * 1.02,
         'Regime→adverse', fontsize=7, color='purple', ha='left', va='bottom', rotation=90)
ax2.set_title('Type B — Cyclical (2022 rate-rise, 2021Q4–present)', fontsize=9, fontweight='bold')
ax2.set_ylabel('Equity (base=1.0 at start)', fontsize=8)
ax2.set_xlabel('Date', fontsize=8)
ax2.tick_params(labelsize=7)
ax2.grid(True, alpha=0.3)
ax2.annotate(
    'Austin only: –12%\n(10 quarters from peak)\nMiami/Phoenix: still rising',
    xy=(pd.Timestamp('2023-01-01'), 1.10), fontsize=7,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8)
)
green_patch = mpatches.Patch(color='green', alpha=0.25, label='Accommodative (regime=1)')
ax2.legend(handles=ax2.get_legend_handles_labels()[0] + [green_patch],
           labels=ax2.get_legend_handles_labels()[1] + ['Accommodative (regime=1)'],
           fontsize=7, loc='upper left')

plt.tight_layout()
fig_path = OUT_DIR / 'phase8_crash_typology.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path}')

Figure saved → /Users/axl/leviathan-system/outputs/oos/phase8_crash_typology.png


/var/folders/dg/2kv5bj0j5d9c8wklbylhzg800000gn/T/ipykernel_28094/3014468366.py:117: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Cell 4 — Conclusion

The `real_rate < 0` regime filter operates as a **systemic risk gate**, not a cyclical
timing tool. In the GFC episode (Type A), the adverse regime (positive real rates,
regime=0) was continuously in place throughout the 2006–2012 period; there was no
regime transition to observe, and the overlay — which only invests during accommodative
quarters — was simply never in the market when the crash arrived, sidestepping
drawdowns of 47–61% in Phoenix, Las Vegas, and Miami at zero opportunity cost (regime
never turned accommodative). In the 2022 episode (Type B), the regime correctly
transitioned back to adverse at 2021Q4 as the Fed tightened, and the overlay again
exited the market — but the subsequent price correction was slow (10–14 quarters from
transition to trough) and shallow (–12% Austin, –4% Las Vegas, –3% Miami, 0% Phoenix),
meaning the correction fell almost entirely outside the 4-quarter crash-label window
used to evaluate signal quality in the OOS notebooks. The signal is directionally
correct in both cases — adverse regimes do precede elevated risk — but the GFC
established the calibration, and the 2022 correction was a different animal: demand-led,
gradual, and market-specific rather than credit-driven, simultaneous, and catastrophic.
Extending the filter to Type B environments would require a longer look-ahead window
(12–16 quarters), city-level supply controls to separate constrained from unconstrained
markets, and an updated DTI fragility check at the cycle peak — none of which are
available in the current single-signal specification.